In [ ]:
"""
═══════════════════════════════════════════════════════════════════════
  BHUTAN VOICE-FIRST PUBLIC SERVICE ASSISTANT
  Conversation + Gemini-powered Request Classifier
  Omdena Sprint · Team 4 deliverable
───────────────────────────────────────────────────────────────────────

  WHAT THIS SCRIPT DOES
  ─────────────────────
  Same conversation output style as Team 3's prototype, PLUS a
  classification table for every user turn showing four dimensions
  with confidence scores:

      ┌──────────────────┬──────────────────────┬────────────┐
      │ In-scope         │ in_scope             │ 0.97       │
      │ Safety           │ safe                 │ 0.99       │
      │ Request type     │ document_inquiry     │ 0.94       │
      │ Service          │ permits              │ 0.98       │
      └──────────────────┴──────────────────────┴────────────┘

  HOW TO RUN IN COLAB
  ───────────────────
    Cell 1:  !pip install -q google-generativeai
    Cell 2:  paste this whole file, set GEMINI_API_KEY env var, run.
    Cell 3:  run_demo()         # runs the 6 sample scenarios
             run_interactive()  # type your own messages
═══════════════════════════════════════════════════════════════════════
"""

# ═══════════════════════════════════════════════════════════════════
# 🔑  STEP 1 — SET GEMINI_API_KEY AS AN ENVIRONMENT VARIABLE
# ═══════════════════════════════════════════════════════════════════
import os
GEMINI_API_KEY    = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL_NAME = "gemini-2.5-flash"   # ← swap for any Gemini model


# ═══════════════════════════════════════════════════════════════════
# 📦  IMPORTS — usually nothing to edit here
# ═══════════════════════════════════════════════════════════════════
import json
import uuid
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

try:
    import google.generativeai as genai
    GENAI_AVAILABLE = True
except ImportError:
    GENAI_AVAILABLE = False


# ═══════════════════════════════════════════════════════════════════
# ✏️  EDIT ME #1 — CLASSIFIER PROMPT
# ═══════════════════════════════════════════════════════════════════
# This is the prompt Gemini sees on every user turn.
# To improve classification accuracy: edit the text below.
# Keep the JSON schema at the bottom intact — the parser depends
# on the four keys: in_scope / safety / request_type / service.
# ═══════════════════════════════════════════════════════════════════
CLASSIFIER_PROMPT_TEMPLATE = """You are a request classifier for a Bhutan public-service voice assistant. \
Classify the user's request along 4 dimensions and return a JSON object.

USER REQUEST: "{user_text}"

DIMENSIONS

1) in_scope — is this about Bhutanese public services?
   • "in_scope"     — permits, health services, business registration, NDI,
                      civil registration, taxes, education, etc.
   • "out_of_scope" — sports, entertainment, weather, jokes, stock prices,
                      general chat, anything else.

2) safety — is this safe to answer with general/procedural information?
   • "safe"   — information, navigation, document checklists, office locations.
   • "unsafe" — requests for medical diagnosis, treatment, prescriptions,
                or specific legal/financial advice.

3) request_type — what kind of help is the user asking for? Pick ONE from:
   {request_types}

4) service — which government domain does this belong to? Pick ONE from:
   {services}

OUTPUT FORMAT — return ONLY this JSON object, nothing else:
{{
  "in_scope":     {{"label": "...", "confidence": 0.0}},
  "safety":       {{"label": "...", "confidence": 0.0}},
  "request_type": {{"label": "...", "confidence": 0.0}},
  "service":      {{"label": "...", "confidence": 0.0}}
}}

Confidence is a number between 0.0 and 1.0 expressing how sure you are."""


# ═══════════════════════════════════════════════════════════════════
# ✏️  EDIT ME #2 — LABEL CATEGORIES
# ═══════════════════════════════════════════════════════════════════
# Add / remove / rename labels here. Gemini will be told to pick
# from these lists.
# ═══════════════════════════════════════════════════════════════════
REQUEST_TYPES = [
    "document_inquiry",     # what documents do I need?
    "status_check",         # check application status
    "office_location",      # where is the office?
    "application_start",    # start a new application
    "eligibility_check",    # am I eligible?
    "general_info",         # explain / general question
    "other",
]

SERVICES = [
    "permits",
    "health",
    "business",
    "general",
]


# ═══════════════════════════════════════════════════════════════════
# 🔌  CLASSIFIER INTERFACE — SWAP IN YOUR OWN NLP HERE
# ═══════════════════════════════════════════════════════════════════
# How to plug in a different classifier (rule-based, BERT, OpenAI, …):
#   1) Subclass BaseClassifier
#   2) Implement classify(user_text) -> Dict[str, ClassificationResult]
#   3) Change the line marked  👉 SWAP CLASSIFIER HERE 👈  at the bottom
# That's it — nothing else in the script needs to change.
# ═══════════════════════════════════════════════════════════════════
@dataclass
class ClassificationResult:
    label: str
    confidence: float


class BaseClassifier:
    """Implement classify() in your subclass."""
    def classify(self, user_text: str) -> Dict[str, ClassificationResult]:
        raise NotImplementedError


class GeminiClassifier(BaseClassifier):
    """One Gemini API call returns all 4 classifications as JSON."""

    def __init__(self, api_key: str, model_name: str = GEMINI_MODEL_NAME):
        if not GENAI_AVAILABLE:
            raise ImportError(
                "google-generativeai not installed.\n"
                "Run this first: !pip install google-generativeai"
            )
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(
            model_name,
            generation_config={"response_mime_type": "application/json"},
        )

    def classify(self, user_text: str) -> Dict[str, ClassificationResult]:
        prompt = CLASSIFIER_PROMPT_TEMPLATE.format(
            user_text=user_text.replace('"', "'"),
            request_types=", ".join(REQUEST_TYPES),
            services=", ".join(SERVICES),
        )
        try:
            response = self.model.generate_content(prompt)
            data = json.loads(response.text)
            return self._parse(data)
        except Exception as e:
            print(f"[classifier error: {type(e).__name__}: {e}]")
            return self._unknown()

    def _parse(self, data: Dict) -> Dict[str, ClassificationResult]:
        out: Dict[str, ClassificationResult] = {}
        for key in ("in_scope", "safety", "request_type", "service"):
            entry = data.get(key, {})
            if not isinstance(entry, dict):
                entry = {}
            out[key] = ClassificationResult(
                label=str(entry.get("label", "unknown")),
                confidence=float(entry.get("confidence", 0.0)),
            )
        return out

    @staticmethod
    def _unknown() -> Dict[str, ClassificationResult]:
        return {k: ClassificationResult("unknown", 0.0)
                for k in ("in_scope", "safety", "request_type", "service")}


class KeywordClassifier(BaseClassifier):
    """
    No-API fallback. Uses the same keyword-matching style as Team 3's
    original code — demonstrates how easy it is to swap in a different
    classifier without changing anything else.
    """

    OUT_OF_SCOPE_WORDS = [
        "cricket", "football", "soccer", "movie", "joke", "weather",
        "stock", "forex", "election", "recipe", "song", "celebrity",
    ]
    UNSAFE_WORDS = [
        "diagnose", "diagnosis", "cancer", "tumour", "tumor",
        "treatment", "prescription", "should i take",
        "what disease", "am i sick", "do i have",
    ]
    SERVICE_KEYWORDS = {
        "permits":  ["permit", "license", "licence", "construction",
                     "timber", "stone", "sand"],
        "health":   ["hospital", "doctor", "clinic", "bhu", "vaccin",
                     "medic", "health", "sick", "pain"],
        "business": ["business", "register", "registration", "company",
                     "proprietor", "trade"],
    }
    TYPE_KEYWORDS = {
        "document_inquiry":  ["document", "papers", "what do i need",
                              "checklist", "requirement"],
        "status_check":      ["status", "track", "reference",
                              "where is my", "progress"],
        "office_location":   ["office", "where", "address", "location",
                              "directions"],
        "application_start": ["apply", "start", "submit", "register",
                              "begin", "new"],
        "eligibility_check": ["eligible", "can i", "qualify", "allowed"],
    }

    def classify(self, text: str) -> Dict[str, ClassificationResult]:
        t = text.lower()
        return {
            "in_scope":     self._in_scope(t),
            "safety":       self._safety(t),
            "service":      self._best_match(t, self.SERVICE_KEYWORDS, "general"),
            "request_type": self._best_match(t, self.TYPE_KEYWORDS, "general_info"),
        }

    def _in_scope(self, t: str) -> ClassificationResult:
        if any(w in t for w in self.OUT_OF_SCOPE_WORDS):
            return ClassificationResult("out_of_scope", 0.8)
        return ClassificationResult("in_scope", 0.7)

    def _safety(self, t: str) -> ClassificationResult:
        if any(w in t for w in self.UNSAFE_WORDS):
            return ClassificationResult("unsafe", 0.8)
        return ClassificationResult("safe", 0.8)

    @staticmethod
    def _best_match(t: str, mapping: Dict[str, List[str]],
                    default: str) -> ClassificationResult:
        scores = {k: sum(1 for w in v if w in t) for k, v in mapping.items()}
        if not any(scores.values()):
            return ClassificationResult(default, 0.4)
        best = max(scores, key=scores.get)
        return ClassificationResult(best, min(1.0, scores[best] / 2.0))


# ═══════════════════════════════════════════════════════════════════
# 📊  TABLE FORMATTER — pretty-prints the classification result
# ═══════════════════════════════════════════════════════════════════
def format_classification_table(c: Dict[str, ClassificationResult]) -> str:
    rows = [
        ("In-scope",     c["in_scope"]),
        ("Safety",       c["safety"]),
        ("Request type", c["request_type"]),
        ("Service",      c["service"]),
    ]
    out = [
        "┌──────────────────┬──────────────────────┬────────────┐",
        "│ Dimension        │ Classification       │ Confidence │",
        "├──────────────────┼──────────────────────┼────────────┤",
    ]
    for name, r in rows:
        label = r.label if len(r.label) <= 20 else r.label[:17] + "..."
        out.append(f"│ {name:<16} │ {label:<20} │ {r.confidence:>10.2f} │")
    out.append("└──────────────────┴──────────────────────┴────────────┘")
    return "\n".join(out)


# ═══════════════════════════════════════════════════════════════════
#                       ━━━━  ENGINE BELOW  ━━━━
# ═══════════════════════════════════════════════════════════════════
# The rest of the file is the conversation engine.
# You usually don't need to edit anything below unless you're
# changing what the assistant SAYS or what knowledge it has.
# ═══════════════════════════════════════════════════════════════════


# ───────────────────────────────────────────────────────────────────
# Knowledge base — small set of chunks per domain. Add more freely.
# ───────────────────────────────────────────────────────────────────
@dataclass
class KnowledgeChunk:
    id: str
    service: str               # "permits" / "health" / "business" / "general"
    title: str
    text: str
    keywords: Tuple[str, ...]
    source: str


KNOWLEDGE_BASE: List[KnowledgeChunk] = [
    # ── PERMITS ─────────────────────────────────────────────────────
    KnowledgeChunk(
        id="prm_construction",
        service="permits",
        title="Construction Permit — Thimphu",
        text=("For a construction permit in Thimphu you will need:\n"
              "  • CID card (original + photocopy)\n"
              "  • Land Thram (land ownership document)\n"
              "  • No Objection Certificate (NOC) from the landlord\n"
              "  • Completed application form from the dzongkhag office\n"
              "Submit at the Department of Urban Development, "
              "Thadrak, Thimphu (Mon–Fri, 9am–5pm)."),
        keywords=("construction", "permit", "thimphu", "document"),
        source="bhutan_flow_docs/permits",
    ),
    KnowledgeChunk(
        id="prm_timber",
        service="permits",
        title="Timber Permit",
        text=("For a timber permit you will need:\n"
              "  • CID card\n"
              "  • Land Thram showing forest plot\n"
              "  • Approval from gewog forest officer\n"
              "  • Completed timber permit form\n"
              "Apply at your local Range Office under the Department of Forests."),
        keywords=("timber", "permit", "forest", "wood"),
        source="bhutan_flow_docs/permits",
    ),
    KnowledgeChunk(
        id="prm_office",
        service="permits",
        title="Permit Office — Thimphu",
        text=("Permit office in Thimphu: Department of Urban Development, "
              "Thadrak, Thimphu. Phone: +975-2-xxx-xxxx. "
              "Open Monday to Friday, 9am to 5pm. "
              "Bring original documents and two photocopies."),
        keywords=("office", "location", "where", "address", "thimphu"),
        source="bhutan_flow_docs/permits",
    ),

    # ── HEALTH ──────────────────────────────────────────────────────
    KnowledgeChunk(
        id="hlt_facility_thimphu",
        service="health",
        title="JDWNRH — Thimphu",
        text=("Nearest hospital in Thimphu: Jigme Dorji Wangchuck National "
              "Referral Hospital (JDWNRH), Gongphel Lam, Thimphu. "
              "Phone: 112 or +975-2-xxx-xxxx. "
              "OPD hours: Mon–Sat 8am–4pm. Emergency: 24 hours."),
        keywords=("hospital", "thimphu", "facility", "nearest"),
        source="bhutan_flow_docs/health",
    ),
    KnowledgeChunk(
        id="hlt_facility_paro",
        service="health",
        title="Paro District Hospital",
        text=("Nearest hospital in Paro: Paro District Hospital, Paro town. "
              "Phone: +975-8-xxx-xxxx. Open Mon–Sat, 8am–4pm."),
        keywords=("hospital", "paro", "facility"),
        source="bhutan_flow_docs/health",
    ),
    KnowledgeChunk(
        id="hlt_bhu",
        service="health",
        title="Basic Health Units (BHUs)",
        text=("Basic Health Units are present in every block of Bhutan. "
              "Services: outpatient consultation, vaccination, antenatal care, "
              "family planning, referral to district hospitals. "
              "All BHU services are free for Bhutanese citizens."),
        keywords=("bhu", "basic health unit", "free", "vaccination"),
        source="bhutan_flow_docs/health",
    ),

    # ── BUSINESS ────────────────────────────────────────────────────
    KnowledgeChunk(
        id="biz_sole",
        service="business",
        title="Sole Proprietorship Registration",
        text=("To register a sole proprietorship in Bhutan you will need:\n"
              "  • Valid CID card\n"
              "  • Completed application form (from MoEA or moea.gov.bt)\n"
              "  • Proof of address\n"
              "  • Initial capital declaration\n"
              "  • No-objection letter if operating from rented premises"),
        keywords=("business", "register", "sole", "proprietorship"),
        source="bhutan_flow_docs/business",
    ),
    KnowledgeChunk(
        id="biz_processing",
        service="business",
        title="License Processing Time",
        text=("If all required information, documents, and clearances are "
              "submitted, the industry license or registration certificate "
              "is issued within one or two working days."),
        keywords=("how long", "processing", "time", "license"),
        source="bhutan_flow_docs/business",
    ),
    KnowledgeChunk(
        id="biz_office",
        service="business",
        title="MoEA Office — Thimphu",
        text=("Business registration office: Ministry of Economic Affairs (MoEA), "
              "Tashichhodzong vicinity, Thimphu. Phone: +975-2-xxx-xxxx. "
              "Open Mon–Fri, 9am–5pm. Online: www.moea.gov.bt"),
        keywords=("office", "moea", "thimphu", "where", "business"),
        source="bhutan_flow_docs/business",
    ),
]


def search_kb(user_text: str, service: str, top_k: int = 2
              ) -> List[KnowledgeChunk]:
    """Simple keyword overlap search filtered by service."""
    t = user_text.lower()
    candidates = [c for c in KNOWLEDGE_BASE if c.service == service]
    if not candidates:
        candidates = KNOWLEDGE_BASE  # fallback to whole KB
    scored = []
    for c in candidates:
        score = sum(1 for kw in c.keywords if kw in t)
        score += sum(1 for word in t.split() if word in c.text.lower()) * 0.1
        scored.append((c, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [c for c, s in scored[:top_k] if s > 0] or candidates[:1]


# ───────────────────────────────────────────────────────────────────
# Conversation engine — produces Team-3-style bot responses
# ───────────────────────────────────────────────────────────────────
@dataclass
class TurnResponse:
    bot_text: str
    sources: List[str]
    step_label: str = ""
    fallback_id: Optional[str] = None
    ref_no: str = ""


class ConversationEngine:
    """
    Single-turn router driven by classifier output.

        out_of_scope  →  FB-03 polite redirect
        unsafe        →  safety redirect (no medical advice)
        else          →  retrieve from KB by service, format response
    """

    SERVICE_INTROS = {
        "permits":  "Great! Let me guide you through that. 🪪",
        "health":   "Of course! Let me guide you to the right information. 🏥",
        "business": "Great! Let me guide you through the process. 🏢",
        "general":  "Let me share what I can help with. 📋",
    }

    def process(self, user_text: str,
                classification: Dict[str, ClassificationResult]
                ) -> TurnResponse:
        ref = f"REF-{uuid.uuid4().hex[:8].upper()}"

        # ── Routing rule 1: out-of-scope ──
        if classification["in_scope"].label == "out_of_scope":
            return TurnResponse(
                bot_text=(
                    "I can help only with the public services in this assistant.\n"
                    "I can still help you find the right office, start again, "
                    "or talk to a person."
                ),
                sources=[],
                fallback_id="FB-03",
                ref_no=ref,
            )

        # ── Routing rule 2: unsafe (medical advice etc.) ──
        if classification["safety"].label == "unsafe":
            return TurnResponse(
                bot_text=(
                    "I can share general health information, but I'm not able to "
                    "give medical advice or diagnose conditions. 🏥\n"
                    "For personal health guidance, please visit your nearest BHU "
                    "or call 112."
                ),
                sources=[],
                fallback_id=None,
                ref_no=ref,
            )

        # ── Routing rule 3: deliver service answer ──
        service = classification["service"].label
        if service not in {"permits", "health", "business"}:
            service = "general"

        chunks = search_kb(user_text, service=service, top_k=2)
        intro  = self.SERVICE_INTROS.get(service, self.SERVICE_INTROS["general"])

        if chunks:
            body    = "\n\n".join(c.text for c in chunks)
            sources = [c.source for c in chunks]
        else:
            body = ("Please visit the relevant dzongkhag office for specific "
                    f"details. Your reference: {ref}")
            sources = []

        bot_text = (
            f"{intro}\n{body}\n"
            "Is there anything else I can help you with?"
        )

        return TurnResponse(
            bot_text=bot_text,
            sources=sources,
            step_label="Step 3 of 4",
            fallback_id=None,
            ref_no=ref,
        )


# ───────────────────────────────────────────────────────────────────
# Output formatter — prints USER + BOT lines, then the table
# ───────────────────────────────────────────────────────────────────
def print_turn(user_text: str, response: TurnResponse,
               classification: Dict[str, ClassificationResult]) -> None:
    print(f"[USER] {user_text}")
    bot_lines = response.bot_text.split("\n")
    print(f"[BOT]  {bot_lines[0]}")
    for line in bot_lines[1:]:
        print(f"       {line}")
    if response.step_label:
        print(f"       {response.step_label}")
    if response.sources:
        print(f"       📚 Sources: {response.sources}")
    if response.fallback_id:
        print(f"       ⚠ Fallback: {response.fallback_id}")
    print()
    print(format_classification_table(classification))
    print()


# ═══════════════════════════════════════════════════════════════════
#                        ━━━━  RUNNERS  ━━━━
# ═══════════════════════════════════════════════════════════════════

DEMO_SCENARIOS = [
    ("normal permit request",
     "I want to know what documents I need for a construction permit in Thimphu"),
    ("out-of-scope request",
     "Who won the cricket match yesterday?"),
    ("unsafe health request",
     "My stomach has been hurting for a week — do I have cancer?"),
    ("normal health request",
     "Where is the nearest hospital in Paro?"),
    ("normal business request",
     "What documents do I need to register a sole proprietorship?"),
    ("out-of-scope chitchat",
     "Can you tell me a joke?"),
]


def make_classifier() -> BaseClassifier:
    """
    👉  SWAP CLASSIFIER HERE  👈
    Replace the line below to use a different classifier:
        return YourCustomClassifier(...)
    """
    if GENAI_AVAILABLE and GEMINI_API_KEY:
        return GeminiClassifier(api_key=GEMINI_API_KEY)
    print("⚠  Using KeywordClassifier fallback "
          "(no Gemini API key set, or library not installed).\n")
    return KeywordClassifier()


def run_demo() -> None:
    """Run the pre-defined sample scenarios."""
    classifier = make_classifier()
    engine     = ConversationEngine()
    print("=" * 72)
    print(" Bhutan Voice-First Public Service Assistant — Demo with Classifier")
    print("=" * 72)
    for label, user_text in DEMO_SCENARIOS:
        print(f"\n── Scenario: {label} ──")
        c = classifier.classify(user_text)
        r = engine.process(user_text, c)
        print_turn(user_text, r, c)


def run_interactive() -> None:
    """Chat with the assistant. Type 'exit' to quit."""
    classifier = make_classifier()
    engine     = ConversationEngine()
    print("=" * 72)
    print(" Bhutan Voice-First Public Service Assistant")
    print("=" * 72)
    print(" Kuzuzangpo la! 🙏  How can I help you today?")
    print(" Type your question below. Type 'exit' (or 'quit') to end the chat.\n")
    while True:
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nTashi Delek! 🙏")
            break
        if not user_text:
            continue                       # ignore empty enter — feels more chat-like
        if user_text.lower() in {"exit", "quit"}:
            print("\nTashi Delek! 🙏")
            break
        print()
        c = classifier.classify(user_text)
        r = engine.process(user_text, c)
        print_turn(user_text, r, c)


# ───────────────────────────────────────────────────────────────────
# Default action: open an interactive chat
# ───────────────────────────────────────────────────────────────────
# Want to see the 6 pre-defined sample scenarios instead?
# Comment out run_interactive() and uncomment run_demo() below.
# ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run_interactive()
    # run_demo()